# Continuous interaction and Coverage90 sensitivity


In [ ]:
import os, json, hashlib, time, warnings
from datetime import datetime
from pathlib import Path

CPU_WORKERS = max(1, int(os.environ.get("CHD_SUPPLEMENT_WORKERS", "12")))
INNER_THREADS = max(1, int(os.environ.get("CHD_INNER_THREADS", "1")))
for key in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[key] = str(INNER_THREADS)

PROJECT_ROOT = Path(os.environ.get("CHD_OUTPUT_ROOT", "../outputs")).resolve()
PRIMARY_RUN_ID = os.environ.get("CHD_PRIMARY_RUN_ID", "final_rerun_2026")
COVERAGE90_RUN_ID = "sensitivity_coverage90"
PRIMARY_RUN_ROOT = PROJECT_ROOT / "runs" / PRIMARY_RUN_ID
COVERAGE90_RUN_ROOT = PROJECT_ROOT / "runs" / COVERAGE90_RUN_ID

NOTEBOOK_DIR = Path.cwd()
NOTEBOOK_PATHS = {
    "shared": Path(os.environ.get("CHD_SHARED_NOTEBOOK", str(NOTEBOOK_DIR / "00_shared_config.ipynb"))),
    "result1": NOTEBOOK_DIR / "01_cohort_and_nmr_derivation.ipynb",
    "result2": NOTEBOOK_DIR / "02_incremental_value.ipynb",
    "result3": NOTEBOOK_DIR / "03_nmr_representation_comparison.ipynb",
    "result5": NOTEBOOK_DIR / "05_calibration_dca_and_joint_risk.ipynb",
}
for path in NOTEBOOK_PATHS.values():
    if not path.is_file():
        raise FileNotFoundError(path)

def stamp(message):
    print(f"{datetime.now().isoformat(sep=' ', timespec='seconds')} | {message}", flush=True)

def run_notebook(path, label):
    started = time.time(); stamp(f"START {label} | {path}")
    get_ipython().run_line_magic("run", f'"{path}"')
    stamp(f"DONE {label} | elapsed={(time.time()-started)/60:.1f} min")

def atomic_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{os.getpid()}.tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(temporary, path)

stamp(f"CPU_WORKERS={CPU_WORKERS}; INNER_THREADS={INNER_THREADS}; GPU=not used (Cox/bootstrap CPU estimands)")
stamp(f"PRIMARY_RUN_ROOT={PRIMARY_RUN_ROOT}")
stamp(f"COVERAGE90_RUN_ROOT={COVERAGE90_RUN_ROOT}")


## Continuous PRS by NMR score interaction


In [ ]:
os.environ["CHD_COHORT_ID"] = "prs_raw106_complete"
os.environ["CHD_RUN_ID"] = PRIMARY_RUN_ID
os.environ["CHD_OUTPUT_ROOT"] = str(PROJECT_ROOT)
os.environ["CHD_SENSITIVITY_OUTPUT_DIR"] = str(PRIMARY_RUN_ROOT)
os.environ["CHD_CPU_WORKERS"] = str(CPU_WORKERS)
os.environ["CHD_INNER_THREADS"] = str(INNER_THREADS)

run_notebook(NOTEBOOK_PATHS["shared"], "Primary shared config")
df_train, df_val, df_test, final_feature_sets, pipeline_manifest = validate_pipeline_artifacts(load_frames=True)
stamp(f"PRIMARY LOCKED | train={len(df_train):,}/{int(df_train.Event.sum()):,} events | val={len(df_val):,}/{int(df_val.Event.sum()):,} | test={len(df_test):,}/{int(df_test.Event.sum()):,}")


In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning
from lifelines.statistics import proportional_hazard_test

INTERACTION_ROOT = Path(RESULTS_OUTPUT_DIR) / "supplementary_prs_nmr_score_interaction"
INTERACTION_ROOT.mkdir(parents=True, exist_ok=True)

interaction_features = list(dict.fromkeys(list(final_feature_sets["Clinical_only"]) + [PRS_COL, "NMR_score"]))
X_train_i, X_val_i, X_test_i, interaction_actual = prepare_X_train_val_test(
    df_train, df_val, df_test, interaction_features, scale=True
)
if PRS_COL not in interaction_actual or "NMR_score" not in interaction_actual:
    raise RuntimeError("PRS or NMR_score missing from continuous interaction design")

split_payloads = {
    "train": (df_train, X_train_i),
    "validation": (df_val, X_val_i),
    "test": (df_test, X_test_i),
}
pipeline_sha = current_pipeline_sha256()

def _interaction_signature(split_name, frame, columns):
    payload = {
        "analysis": "continuous_PRS_x_NMR_score",
        "split": split_name,
        "pipeline_manifest_sha256": pipeline_sha,
        "eid_sha256": eid_digest(frame),
        "outcome_sha256": dataframe_digest(frame, ["Eid", "Time", "Event"]),
        "columns": list(columns) + ["PRS_x_NMR_score"],
        "preprocessing": "training-only prepare_X_train_val_test(scale=True)",
        "model": "unpenalized Cox PH; robust sandwich SE; Efron ties; batch_mode",
    }
    signature = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    return payload, signature

def _fit_interaction_split(split_name):
    frame, design = split_payloads[split_name]
    design = design.reset_index(drop=True).copy()
    design["PRS_x_NMR_score"] = design[PRS_COL].to_numpy(float) * design["NMR_score"].to_numpy(float)
    fit_frame = pd.concat([frame[["Time","Event"]].reset_index(drop=True), design], axis=1)
    payload, signature = _interaction_signature(split_name, frame, design.columns[:-1])
    csv_path = INTERACTION_ROOT / f"continuous_prs_x_nmr_score_{split_name}.csv"
    json_path = INTERACTION_ROOT / f"continuous_prs_x_nmr_score_{split_name}.json"
    if csv_path.exists() and json_path.exists():
        saved = json.loads(json_path.read_text(encoding="utf-8"))
        if saved.get("signature") == signature:
            stamp(f"RESUME interaction | split={split_name} | signed cache valid")
            return pd.read_csv(csv_path).iloc[0].to_dict()

    started = time.time(); stamp(f"START interaction Cox | split={split_name} | N={len(frame):,} | events={int(frame.Event.sum()):,} | features={design.shape[1]}")
    model = CoxPHFitter(penalizer=0.0, l1_ratio=0.0)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        model.fit(fit_frame, duration_col="Time", event_col="Event", robust=True, batch_mode=True, show_progress=False)
    term = model.summary.loc["PRS_x_NMR_score"]

    # PH diagnostic is required only for the locked-test association model.
    if split_name == "test":
        ph = proportional_hazard_test(model, fit_frame, time_transform="rank").summary
        ph_p = float(ph.loc["PRS_x_NMR_score", "p"])
        ph_n = int(ph["p"].notna().sum())
        ph_bonferroni = min(1.0, ph_p * ph_n)
    else:
        ph_p, ph_n, ph_bonferroni = np.nan, 0, np.nan
    row = {
        "Split": split_name, "N": int(len(frame)), "Event_N": int(frame.Event.sum()),
        "Term": "PRS_x_NMR_score", "Beta": float(term["coef"]), "SE": float(term["se(coef)"]),
        "HR": float(term["exp(coef)"]), "HR_CI_Lower": float(term["exp(coef) lower 95%"]),
        "HR_CI_Upper": float(term["exp(coef) upper 95%"]), "P_Value": float(term["p"]),
        "PH_P_Raw": ph_p, "PH_Test_N": ph_n, "PH_P_Bonferroni": ph_bonferroni,
        "Scale": "interaction between training-standardized PRS and training-standardized NMR_score",
        "Adjustment": "all Clinical_only columns plus PRS and NMR_score main effects",
        "Pipeline_Manifest_SHA256": pipeline_sha, "Signature": signature,
        "Elapsed_Minutes": (time.time()-started)/60,
    }
    pd.DataFrame([row]).to_csv(csv_path, index=False)
    atomic_json(json_path, {"signature": signature, "signature_payload": payload})
    stamp(f"DONE interaction Cox | split={split_name} | HR={row['HR']:.5f} | P={row['P_Value']:.4g} | elapsed={row['Elapsed_Minutes']:.1f} min")
    return row

interaction_splits = ["train", "validation", "test"]
interaction_rows = Parallel(
    n_jobs=min(3, CPU_WORKERS), backend="loky", verbose=10,
    pre_dispatch=min(3, CPU_WORKERS), batch_size=1, max_nbytes="50M",
)(delayed(_fit_interaction_split)(split_name) for split_name in interaction_splits)

interaction_results = pd.DataFrame(interaction_rows)
interaction_results.to_csv(INTERACTION_ROOT / "continuous_prs_x_nmr_score_all_splits.csv", index=False)
display(interaction_results[["Split","N","Event_N","HR","HR_CI_Lower","HR_CI_Upper","P_Value","PH_P_Bonferroni","Elapsed_Minutes"]])
stamp(f"INTERACTION COMPLETE | {INTERACTION_ROOT}")


## Coverage90 sensitivity cohort


In [ ]:
os.environ["CHD_COHORT_ID"] = "coverage90"
os.environ["CHD_RUN_ID"] = COVERAGE90_RUN_ID
os.environ["CHD_OUTPUT_ROOT"] = str(PROJECT_ROOT)
os.environ["CHD_SENSITIVITY_OUTPUT_DIR"] = str(COVERAGE90_RUN_ROOT)
os.environ["CHD_CPU_WORKERS"] = str(CPU_WORKERS)
os.environ["CHD_INNER_THREADS"] = str(INNER_THREADS)
os.environ["CHD_BOOTSTRAP_N"] = "1000"
os.environ["CHD_RESULT2_COX_N_JOBS"] = str(min(4, CPU_WORKERS))
os.environ["CHD_RESULT2_BOOTSTRAP_N_JOBS"] = str(min(4, CPU_WORKERS))

run_notebook(NOTEBOOK_PATHS["shared"], "Coverage90 shared config")

BOOTSTRAP_PROGRESS_EVERY = 25
stamp(f"COVERAGE90 CONFIGURED | scenario={MAIN_SCENARIO_NAME} | output={SENSITIVITY_OUTPUT_DIR}")


In [ ]:
stamp("Starting Coverage90 cohort construction and nested score selection.")
run_notebook(NOTEBOOK_PATHS["result1"], "Coverage90 Result 1")

df_train, df_val, df_test, final_feature_sets, pipeline_manifest = validate_pipeline_artifacts(load_frames=True)
if MAIN_SCENARIO_NAME != "incident_chd_coverage90":
    raise RuntimeError(f"Wrong coverage scenario after Result 1: {MAIN_SCENARIO_NAME}")
coverage_audit = pd.DataFrame([
    {"Split": "train", "N": len(df_train), "Event_N": int(df_train.Event.sum())},
    {"Split": "validation", "N": len(df_val), "Event_N": int(df_val.Event.sum())},
    {"Split": "test", "N": len(df_test), "Event_N": int(df_test.Event.sum())},
])
coverage_audit.to_csv(Path(RESULTS_OUTPUT_DIR) / "coverage90_split_audit.csv", index=False)
display(coverage_audit)
stamp(f"Coverage90 Result 1 verified | total N={coverage_audit.N.sum():,} | events={coverage_audit.Event_N.sum():,}")


In [ ]:
stamp("Coverage90 Result 2 starts. Existing signed caches resume automatically.")
stamp("Parallel task messages show completed comparisons; bootstrap heartbeat target is every 25 resamples when supported by the upstream function.")
run_notebook(NOTEBOOK_PATHS["result2"], "Coverage90 Result 2")
stamp("Coverage90 minimum required sensitivity analysis COMPLETE")


In [ ]:
required_names = [
    "result2_cox_model_performance.csv",
    "figure2A_delta_uno_cindex.csv",
    "figure2C_delta_ipcw_ibs.csv",
    "figure2D_ipcw_nri_idi.csv",
]

found = {}

for name in required_names:
    matches = sorted(Path(RESULTS_OUTPUT_DIR).rglob(name))

    if len(matches) == 1:
        found[name] = str(matches[0])
        print(f"FOUND: {name}")
        print(f"       {matches[0]}")
    else:
        found[name] = None
        print(f"MISSING/AMBIGUOUS: {name}, found {len(matches)} files")

minimum_ok = all(found.values())

success_payload = {
    "status": "SUCCESS" if minimum_ok else "INCOMPLETE",
    "analysis": "coverage90 Result1+Result2 sensitivity",
    "run_id": COVERAGE90_RUN_ID,
    "scenario": MAIN_SCENARIO_NAME,
    "timestamp": datetime.now().isoformat(),
    "pipeline_manifest_sha256": current_pipeline_sha256(),
    "train_n": int(len(df_train)),
    "validation_n": int(len(df_val)),
    "test_n": int(len(df_test)),
    "outputs": found,
}

success_path = (
    Path(RESULTS_OUTPUT_DIR)
    / "COVERAGE90_RESULT2_SUCCESS.json"
)

atomic_json(success_path, success_payload)

print("\n" + json.dumps(
    success_payload,
    ensure_ascii=False,
    indent=2,
))

if not minimum_ok:
    raise RuntimeError(
        "Coverage90 Result 2 outputs are incomplete; "
        "see the missing file messages above."
    )

print("SUCCESS marker:", success_path)


In [ ]:
stamp("START optional coverage90 Result 3 representation sensitivity")
run_notebook(NOTEBOOK_PATHS["result3"], "Coverage90 Result 3")
stamp("DONE optional coverage90 Result 3")


In [ ]:
stamp("START optional coverage90 Result 5 calibration/DCA sensitivity")
run_notebook(NOTEBOOK_PATHS["result5"], "Coverage90 Result 5")
stamp("DONE optional coverage90 Result 5")
